### Müllabfuhr – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021/2022), 2. Runde, Aufgabe 1*

Aufstand in Müllhausen! Die städtische Müllabfuhr CityTrash, die leider nur über genau ein Müllauto verfügt, erhält immer mehr
Beschwerden von Anwohnern, dass an deren Straßen die Mülleimer nicht oft genug geleert werden. Um Abhilfe zu schaffen, soll jetzt
jede Straße mindestens einmal pro Woche abgefahren werden.

Hierzu soll ein Wochenfahrplan erstellt werden, der für jeden Wochentag zwischen Montag und Freitag festlegt, welche Straßen das
Müllauto abfahren soll. Dabei kann eine Straße an einem Tag auch mehrmals befahren werden. Da die Straßen von Müllhausen sehr eng
sind, kann der Fahrer nur an Kreuzungen oder am Ende von Sackgassen wenden; selbst Rückwärtsfahren ist zu gefährlich. Jede
Tagesroute beginnt und endet in der Zentrale von CityTrash. Um Überstunden zu vermeiden, soll die maximale Tagesstrecke möglichst
klein gehalten werden.

Ein Beispiel: Wenn Müllhausen so einen einfachen Straßenplan hätte wie im Bild auf dem Aufgabenblatt und alle Straßen 1 km lang
wären, dann kann das Müllauto von Montag bis Donnerstag jeweils einen der „Straßenquadranten“ und am Freitag die beiden Straßen
nach Osten und zurück abfahren. Alle Tagesstrecken und somit auch die maximale Tagesstrecke sind 4 km lang. Dieser Wochenfahrplan
ist optimal.

#### Aufgabe

Hilf CityTrash, indem du ein Programm schreibst, das eine Karte einliest und einen passenden Wochenfahrplan erzeugt. Auf der Karte
sind die Straßen von Müllhausen und die Zentrale von CityTrash eingezeichnet. Wende dein Programm mindestens auf alle Beispiele an,
die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `muellabfuhr0.txt` bis `muellabfuhr8.txt` und `muellabfuhr0-ans.txt` (ein Beispielergebnis zu
`muellabfuhr0.txt`). Die erste Zeile einer Eingabe enthält die Zahl `n` der Kreuzungen und die Zahl `m` der Straßen, danach folgt
pro Straße eine Zeile `u v länge`. Die Kreuzungen heißen `0` bis `n-1`, die Zentrale ist die Kreuzung `0`. Das Format haben wir aus den
Dateien abgelesen. Das Beispielergebnis zeigt, wie die Ausgabe aussehen kann:
`Tag 1: 0 -> 8 -> 9 -> 8 -> 0, Gesamtlaenge: 4`.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Graphen:** Kreuzungen sind Knoten, Straßen sind Kanten mit einer Länge. Man speichert für jeden Knoten die Liste seiner Nachbarn
  (mit Straßenlänge).
- **Kürzeste Wege mit Dijkstra:** Von einem Knoten aus die Abstände zu allen anderen finden. Dazu braucht man eine Warteschlange, die
  immer den nächsten Knoten liefert.
- **Eulerkreis:** Ein Rundweg, der jede Straße **genau einmal** befährt, existiert genau dann, wenn jede Kreuzung eine **gerade** Zahl
  von Straßen hat (jedes Mal, wenn man hineinfährt, muss man wieder hinaus). Der Algorithmus von Hierholzer findet ihn.
- **Knoten ungeraden Grades paaren:** Gibt es Kreuzungen mit ungerader Straßenzahl, dann müssen manche Straßen doppelt befahren werden.
  Verbindet man je zwei ungerade Kreuzungen durch einen kürzesten Weg und fährt diese Wege doppelt, ist alles gerade. Welche Paare die
  kürzesten Zusatzwege ergeben, ist ein *Matching*-Problem.
- **Aufteilen mit binärer Suche:** Gegeben eine Obergrenze `x` für die Tageslänge, kann man den Rundweg einfach von vorn in Tagesstücke
  zerlegen. Die kleinste Obergrenze, bei der es klappt, findet man mit binärer Suche.
- **Zufall und Probieren:** Es gibt viele Eulerkreise. Viele zufällige ausprobieren und den besten nehmen, verbessert das Ergebnis.
- **NP-schwer:** Schon mit zwei Tagen ist die Aufgabe so schwer wie das Aufteilen von Zahlen in zwei gleich große Summen. Eine
  optimale Lösung für die großen Beispiele ist deshalb nicht zu erwarten; wir suchen gute.

**Nützliche Funktionen und Bibliotheken**
- `heapq.heappush` und `heapq.heappop`: Warteschlange, die immer das kleinste Element liefert (für Dijkstra)
- `itertools.combinations(liste, k)`: alle Auswahlen von `k` Elementen
- `random.Random(1)`: Zufallsgenerator mit festem Startwert, damit die Ergebnisse jedes Mal gleich sind; `rng.shuffle(liste)` mischt
- **Sortieren von Tupeln:** `sorted` vergleicht Tupel zuerst nach dem ersten Eintrag, dann nach dem zweiten. Mit `(abstand, a, b)` sortiert
  man Paare nach dem Abstand.

## Die Idee

1. **Alle Straßen in einem Rundweg:** Zuerst vergessen wir die fünf Tage und suchen *einen* kurzen Rundweg ab der Zentrale, der jede
   Straße mindestens einmal befährt. Haben alle Kreuzungen gerade Grade, ist das ein Eulerkreis. Sonst paaren wir die Kreuzungen mit
   ungeradem Grad und fahren die kürzesten Wege zwischen den Partnern ein zweites Mal.
2. **In fünf Stücke teilen:** Den Rundweg zerlegen wir in fünf zusammenhängende Stücke. Jedes Stück fährt an einem Tag: von der
   Zentrale auf einem kürzesten Weg zum Anfang des Stücks, das Stück, dann auf einem kürzesten Weg zurück. Die **Tageslänge** ist
   also Weg hin + Stück + Weg zurück. Mit binärer Suche suchen wir die kleinste Obergrenze, bei der fünf Stücke reichen.
3. **Verbessern:** Das Ergebnis hängt von der Paarung, vom gewählten Eulerkreis und vom Startpunkt der Zerlegung ab. Wir probieren
   mehrere und kürzen am Schluss die Tagesenden ab, wenn dort Straßen nur ein zweites Mal befahren werden.

Eine **untere Schranke** liefert die Länge des Rundwegs: Fügt man die fünf Tagesrouten einer optimalen Lösung aneinander, entsteht ein
Rundweg, der alle Straßen befährt. Er ist nicht kürzer als der kürzeste solche Rundweg (Eulerkreis mit bester Paarung). Also ist die
längste Tagesroute mindestens ein Fünftel davon. Außerdem muss eine Tagesroute die am weitesten entfernte Straße erreichen und
zurückkehren. Die untere Schranke sagt uns, wie weit wir höchstens vom Optimum entfernt sind; sie ist selbst oft nicht erreichbar.

## Schritt 1: Einlesen und der Straßenplan

In [ ]:
import heapq
import itertools
import math
import random


def lies(nr):
    """Liest muellabfuhr<nr>.txt: n Kreuzungen, Liste der Straßen (u, v, länge)."""
    with open('beispieldaten/muellabfuhr%d.txt' % nr, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n, m = [int(z) for z in zeilen[0].split()]
    kanten = []
    for zeile in zeilen[1:]:
        if zeile.strip():
            u, v, laenge = [int(z) for z in zeile.split()]
            kanten.append((u, v, laenge))
    assert len(kanten) == m
    return n, kanten


def straßenplan(n, kanten):
    """Nachbarlisten und ein Wörterbuch der Straßenlängen (in beiden Richtungen)."""
    nachbarn = [[] for _ in range(n)]
    laenge = {}
    for u, v, l in kanten:
        nachbarn[u].append((v, l))
        nachbarn[v].append((u, l))
        laenge[(u, v)] = l
        laenge[(v, u)] = l
    return nachbarn, laenge


n0, kanten0 = lies(0)
print(n0, 'Kreuzungen,', len(kanten0), 'Straßen')
print(kanten0[:4], '...')

10 Kreuzungen, 13 Straßen
[(0, 2, 1), (0, 4, 1), (0, 6, 1), (0, 8, 1)] ...


## Schritt 2: Kürzeste Wege (Dijkstra)

Wir brauchen die kürzesten Wege von der Zentrale zu allen Kreuzungen (für den Weg zum Tagesstück und zurück) und von jeder Kreuzung mit
ungeradem Grad zu allen anderen (für die Paarung). Die Funktion liefert die Abstände `d` und für jeden Knoten den Vorgänger `vor` auf
dem kürzesten Weg, mit dem man den Weg selbst zurückverfolgen kann.

In [ ]:
def dijkstra(nachbarn, start):
    """Abstände von start zu allen Knoten und der Vorgänger jedes Knotens auf dem kürzesten Weg."""
    d = [float('inf')] * len(nachbarn)
    vor = [None] * len(nachbarn)
    d[start] = 0
    warteschlange = [(0, start)]
    while warteschlange:
        abstand, u = heapq.heappop(warteschlange)
        if abstand > d[u]:
            continue                      # veralteter Eintrag
        for v, l in nachbarn[u]:
            if abstand + l < d[v]:
                d[v] = abstand + l
                vor[v] = u
                heapq.heappush(warteschlange, (d[v], v))
    return d, vor


def weg(vor, start, ziel):
    """Kürzester Weg als Knotenliste, aus den Vorgängern von dijkstra(start) zurückverfolgt."""
    w = [ziel]
    while w[-1] != start:
        w.append(vor[w[-1]])
    return w[::-1]


nachbarn0, laenge0 = straßenplan(n0, kanten0)
d0, vor0 = dijkstra(nachbarn0, 0)
print('Abstände von der Zentrale:', d0)
print('Weg von 0 nach 9:', weg(vor0, 0, 9))

Abstände von der Zentrale: [0, 2, 1, 2, 1, 2, 1, 2, 1, 2]
Weg von 0 nach 9: [0, 8, 9]


## Schritt 3: Ungerade Kreuzungen paaren

Der Grad einer Kreuzung ist die Zahl der Straßen dort. Die Kreuzungen mit ungeradem Grad (es sind immer gerade viele) müssen paarweise
durch einen zweiten Weg verbunden werden. **Gierig:** Wir sortieren alle Paare nach ihrem Abstand und nehmen immer das nächste Paar,
bei dem beide Kreuzungen noch frei sind. Das ist einfach, aber nicht immer die beste Paarung.

In [ ]:
def ungerade_knoten(nachbarn):
    return [v for v in range(len(nachbarn)) if len(nachbarn[v]) % 2 == 1]


def abstaende_der_ungeraden(nachbarn):
    """Dijkstra von jedem ungeraden Knoten aus: Abstände und Vorgänger."""
    dist, vor = {}, {}
    for a in ungerade_knoten(nachbarn):
        dist[a], vor[a] = dijkstra(nachbarn, a)
    return dist, vor


def paarung_gierig(ungerade, dist):
    """Immer das Paar mit dem kleinsten Abstand, dessen Knoten beide noch frei sind."""
    alle = sorted((dist[a][b], a, b) for i, a in enumerate(ungerade) for b in ungerade[i + 1:])
    vergeben = set()
    paare = []
    for abstand, a, b in alle:
        if a not in vergeben and b not in vergeben:
            vergeben.add(a)
            vergeben.add(b)
            paare.append([a, b])
    return paare


def paarung_kosten(paare, dist):
    return sum(dist[a][b] for a, b in paare)


ung0 = ungerade_knoten(nachbarn0)
dist0, vorU0 = abstaende_der_ungeraden(nachbarn0)
paare0 = paarung_gierig(ung0, dist0)
print('ungerade Kreuzungen:', ung0)
print('Paare:', paare0, ' Zusatzlänge:', paarung_kosten(paare0, dist0))

ungerade Kreuzungen: [2, 4, 6, 9]
Paare: [[2, 4], [6, 9]]  Zusatzlänge: 5


## Schritt 4: Der Rundweg (Eulerkreis nach Hierholzer)

Wir legen zu jedem Paar die Straßen des kürzesten Weges als **zusätzliche Kanten** (Doppelkanten) in den Plan. Dann haben alle
Kreuzungen geraden Grad. Hierholzer: Man fährt von der Zentrale los und immer über eine noch nicht benutzte Straße weiter (Stapel).
Kann man an einer Kreuzung nicht weiter, kommt sie ans Ende der Ergebnisliste und man geht auf dem Stapel einen Schritt zurück.
Am Ende steht die Ergebnisliste rückwärts. Mit einem Zufallsgenerator `rng` wird die Reihenfolge der Straßen an jeder Kreuzung
gemischt, so erhält man verschiedene Eulerkreise.

In [ ]:
def eulerkreis(n, mehrfachkanten, start, rng=None):
    """Knotenliste eines Rundwegs ab start, der jede Kante der Liste (u, v) genau einmal benutzt."""
    anliegend = [[] for _ in range(n)]
    for i, (u, v) in enumerate(mehrfachkanten):
        anliegend[u].append((i, v))
        anliegend[v].append((i, u))
    if rng is not None:
        for liste in anliegend:
            rng.shuffle(liste)
    benutzt = [False] * len(mehrfachkanten)
    stapel = [start]
    ergebnis = []
    while stapel:
        u = stapel[-1]
        while anliegend[u] and benutzt[anliegend[u][-1][0]]:
            anliegend[u].pop()
        if anliegend[u]:
            i, v = anliegend[u].pop()
            benutzt[i] = True
            stapel.append(v)
        else:
            ergebnis.append(stapel.pop())
    return ergebnis[::-1]


def rundweg(n, kanten, paare, vorU, rng=None):
    """Alle Straßen plus die Wege zwischen den Paaren (doppelt befahren) als ein Rundweg ab der Zentrale 0."""
    mehrfach = [(u, v) for u, v, l in kanten]
    for a, b in paare:
        w = weg(vorU[a], a, b)
        mehrfach += list(zip(w, w[1:]))
    return eulerkreis(n, mehrfach, 0, rng)


def weglaenge(w, laenge):
    return sum(laenge[(a, b)] for a, b in zip(w, w[1:]))


rund0 = rundweg(n0, kanten0, paare0, vorU0)
print(rund0)
print('Länge des Rundwegs:', weglaenge(rund0, laenge0), '= 13 + 5 Zusatzlänge, geteilt durch 5 ergibt aufgerundet',
      math.ceil(weglaenge(rund0, laenge0) / 5))

[0, 8, 9, 8, 1, 2, 0, 6, 7, 8, 0, 4, 5, 6, 0, 4, 3, 2, 0]
Länge des Rundwegs: 18 = 13 + 5 Zusatzlänge, geteilt durch 5 ergibt aufgerundet 4


## Schritt 5: In fünf Tage teilen

Zuerst die einfachere Frage: **Reicht die Obergrenze `x`?** Wir starten am Anfang des Rundwegs und nehmen das längste Stück, dessen
Tageslänge (Weg von der Zentrale zum Anfang + Stück + Weg zurück vom Ende) höchstens `x` ist. Das nächste Stück beginnt dort, wo das
vorige endete. Sind nach fünf Stücken alle Straßen des Rundwegs dran, reicht `x`.

Dass dieses gierige Vorgehen nicht immer die beste Zerlegung findet (der Rückweg zur Zentrale ist von Kreuzung zu Kreuzung
verschieden lang), nehmen wir in Kauf. Der Parameter `dreh` lässt den Rundweg an einer anderen Stelle beginnen. Das ist erlaubt,
denn jedes Stück wird ohnehin über kürzeste Wege mit der Zentrale verbunden.

In [ ]:
def zerlege(w, laenge, d0, x, dreh=0):
    """Teilt den Rundweg w (Knotenliste, w[0] == w[-1]), der bei Position dreh beginnt, gierig in höchstens fünf Stücke
    mit Tageslänge <= x. Ergibt (gedrehter Weg, Liste der Stücke (von, bis)) oder None."""
    k = len(w) - 1
    ww = w[dreh:k] + w[:dreh] + [w[dreh]]
    summe = [0]                                   # summe[i]: Länge des Weges bis Position i
    for a, b in zip(ww, ww[1:]):
        summe.append(summe[-1] + laenge[(a, b)])
    i = 0
    stuecke = []
    for tag in range(5):
        if i == k:
            break
        beste = None
        j = i + 1
        while j <= k and d0[ww[i]] + summe[j] - summe[i] <= x:
            if d0[ww[i]] + summe[j] - summe[i] + d0[ww[j]] <= x:
                beste = j                         # so weit kommt man noch und ist am Abend wieder in der Zentrale
            j += 1
        if beste is None:
            return None
        stuecke.append((i, beste))
        i = beste
    if i < k:
        return None
    while len(stuecke) < 5:
        stuecke.append((k, k))                    # freie Tage
    return ww, stuecke


def beste_zerlegung(w, laenge, d0, dreh=0):
    """Binäre Suche nach der kleinsten Obergrenze x, für die zerlege klappt."""
    unten = 0
    oben = weglaenge(w, laenge) + 2 * max(d0)    # so viel reicht bestimmt (alles an einem Tag)
    while unten < oben:
        mitte = (unten + oben) // 2
        if zerlege(w, laenge, d0, mitte, dreh) is not None:
            oben = mitte
        else:
            unten = mitte + 1
    return unten, zerlege(w, laenge, d0, unten, dreh)


x0, (ww0, stuecke0) = beste_zerlegung(rund0, laenge0, d0)
print('kleinste Obergrenze:', x0)
print('Stücke (von Position, bis Position):', stuecke0)

kleinste Obergrenze: 4
Stücke (von Position, bis Position): [(0, 3), (3, 6), (6, 10), (10, 14), (14, 18)]


## Schritt 6: Tagesrouten, Prüfung und Ausgabe

Aus jedem Stück wird eine Tagesroute: kürzester Weg von der Zentrale zum Anfang, das Stück, kürzester Weg zum Ende zurück. Eine
Funktion `pruefe` stellt sicher, dass jede Route in der Zentrale beginnt und endet, nur über vorhandene Straßen führt und dass **jede
Straße** an mindestens einem Tag vorkommt. Sie liefert die maximale Tageslänge.

In [ ]:
def tagesrouten(ww, stuecke, vor0):
    routen = []
    for i, j in stuecke:
        if i == j:
            routen.append([0])                    # freier Tag
        else:
            hin = weg(vor0, 0, ww[i])
            zurueck = weg(vor0, 0, ww[j])[::-1]
            routen.append(hin + ww[i + 1:j + 1] + zurueck[1:])
    return routen


def pruefe(routen, kanten, laenge):
    """Ist der Wochenfahrplan gültig? Ergibt die maximale Tageslänge."""
    assert len(routen) == 5
    befahren = set()
    for r in routen:
        assert r[0] == 0 and r[-1] == 0
        for a, b in zip(r, r[1:]):
            assert (a, b) in laenge, (a, b)
            befahren.add((min(a, b), max(a, b)))
    assert befahren == {(min(u, v), max(u, v)) for u, v, l in kanten}, 'nicht alle Straßen'
    return max(weglaenge(r, laenge) for r in routen)


def zeige(routen, laenge, nur_maximum=False):
    for tag, r in enumerate(routen, 1):
        if not nur_maximum:
            print('Tag %d: %s, Gesamtlaenge: %d' % (tag, ' -> '.join(str(k) for k in r), weglaenge(r, laenge)))
    print('Maximale Laenge einer Tagestour:', max(weglaenge(r, laenge) for r in routen))


routen0 = tagesrouten(ww0, stuecke0, vor0)
pruefe(routen0, kanten0, laenge0)
zeige(routen0, laenge0)

Tag 1: 0 -> 8 -> 9 -> 8 -> 0, Gesamtlaenge: 4
Tag 2: 0 -> 8 -> 1 -> 2 -> 0, Gesamtlaenge: 4
Tag 3: 0 -> 6 -> 7 -> 8 -> 0, Gesamtlaenge: 4
Tag 4: 0 -> 4 -> 5 -> 6 -> 0, Gesamtlaenge: 4
Tag 5: 0 -> 4 -> 3 -> 2 -> 0, Gesamtlaenge: 4
Maximale Laenge einer Tagestour: 4


Das ist ein gültiger Wochenfahrplan mit maximaler Tageslänge 4, und er ist **optimal**: Der Rundweg ist 18 lang (13 Straßen plus
5 für die Zusatzwege), ein Fünftel davon ist aufgerundet 4, und kürzer als 4 kann der längste Tag nicht werden.

## Schritt 7: Alles zusammen, erste Version

Die Funktion `loese` fasst die Schritte zusammen. Mit den Standardwerten ist es die einfache Version: gierige Paarung, **ein**
Eulerkreis, Beginn bei der Zentrale. Sie liefert auch die Länge des Rundwegs, aus der wir die untere Schranke berechnen.
Die Parameter `versuche` (so viele zufällige Eulerkreise), `drehungen` (so viele Anfangspunkte pro Eulerkreis), `kuerzen` und
`paarung` schalten die Verbesserungen ein, die wir uns gleich ansehen.

In [ ]:
def loese(nr, versuche=1, drehungen=1, kuerzen=False, paarung='gierig'):
    n, kanten = lies(nr)
    nachbarn, laenge = straßenplan(n, kanten)
    d0, vor0 = dijkstra(nachbarn, 0)
    ung = ungerade_knoten(nachbarn)
    dist, vorU = abstaende_der_ungeraden(nachbarn)
    paare = paarung_gierig(ung, dist)
    if paarung == 'gut':
        paare = verbessere_paarung(paare, dist)
    rng = random.Random(1)
    beste = None
    for versuch in range(versuche):
        w = rundweg(n, kanten, paare, vorU, rng if versuch > 0 else None)
        for dreh in [0] + [rng.randrange(len(w) - 1) for _ in range(drehungen - 1)]:
            x, (ww, stuecke) = beste_zerlegung(w, laenge, d0, dreh)
            routen = tagesrouten(ww, stuecke, vor0)
            maxlaenge = max(weglaenge(r, laenge) for r in routen)
            if kuerzen and (beste is None or maxlaenge <= beste[0] * 1.01):
                routen = kuerze_enden(routen, laenge, d0, vor0)
                maxlaenge = max(weglaenge(r, laenge) for r in routen)
            if beste is None or maxlaenge < beste[0]:
                beste = (maxlaenge, routen)
    rundlaenge = sum(l for u, v, l in kanten) + paarung_kosten(paare, dist)
    assert pruefe(beste[1], kanten, laenge) == beste[0]
    return beste[1], rundlaenge


def verbessere_paarung(paare, dist):
    return paare       # kommt in Schritt 8


def kuerze_enden(routen, laenge, d0, vor0):
    return routen      # kommt in Schritt 8


einfach = {}
for nr in range(9):
    routen, rundlaenge = loese(nr)
    einfach[nr] = max(weglaenge(r, straßenplan(*lies(nr))[1]) for r in routen)
print('längste Tagesstrecke, einfache Version:')
print(einfach)

längste Tagesstrecke, einfache Version:
{0: 4, 1: 21, 2: 11, 3: 23, 4: 10, 5: 1471, 6: 580943, 7: 795073, 8: 2780226}


## Schritt 8: Verbesserungen

**a) Bessere Paarung.** Die gierige Paarung kann man verbessern: Man nimmt zwei Paare, `(a, b)` und `(c, d)`, und prüft die beiden
anderen Möglichkeiten `(a, c), (b, d)` und `(a, d), (b, c)`. Ist eine kürzer, tauscht man. Das genügt oft noch nicht, daher nehmen wir
auch **drei** Paare (sechs Knoten, 15 Möglichkeiten). Das wiederholen wir, bis sich nichts mehr ändert. Die Hilfsfunktion
`alle_paarungen` zählt per Rekursion alle Möglichkeiten auf, sechs Knoten in drei Paare zu teilen.

**b) Viele Eulerkreise und Anfangspunkte.** Jeder zufällig gemischte Eulerkreis (und jeder andere Anfangspunkt) führt zu anderen
Stücken; wir behalten den besten.

**c) Enden kürzen.** Ein Tag fährt am Anfang vielleicht erst eine Straße, die schon an einem anderen Tag befahren wird, obwohl von der
Zentrale ein kürzerer Weg zu der Stelle führt. Dann ersetzen wir den Anfang durch den kürzesten Weg. Das gleiche gilt für das Ende.
Dazu zählen wir, wie oft jede Straße insgesamt befahren wird, und kürzen nur, wenn **keine Straße unbefahren** bleibt. Wir probieren
pro Tagesende die fünf größten Ersparnisse.

In [ ]:
def alle_paarungen(knoten):
    """Alle Möglichkeiten, die Liste in Paare zu teilen (als Liste von Paarlisten)."""
    if not knoten:
        return [[]]
    erster = knoten[0]
    ergebnis = []
    for i in range(1, len(knoten)):
        rest = knoten[1:i] + knoten[i + 1:]
        for p in alle_paarungen(rest):
            ergebnis.append([(erster, knoten[i])] + p)
    return ergebnis


MUSTER = {2: alle_paarungen([0, 1, 2, 3]), 3: alle_paarungen([0, 1, 2, 3, 4, 5])}
print(len(MUSTER[2]), 'Möglichkeiten für 4 Knoten,', len(MUSTER[3]), 'für 6 Knoten')


def verbessere_paarung(paare, dist):
    """Tauscht Partner innerhalb von zwei und drei Paaren, solange das die Zusatzlänge verkürzt."""
    paare = [list(p) for p in paare]
    verbessert = True
    while verbessert:
        verbessert = False
        for k in (2, 3):
            for auswahl in itertools.combinations(range(len(paare)), k):
                knoten = []
                for i in auswahl:
                    knoten += paare[i]
                jetzt = sum(dist[paare[i][0]][paare[i][1]] for i in auswahl)
                beste_kosten, beste_muster = jetzt, None
                for muster in MUSTER[k]:
                    kosten = sum(dist[knoten[a]][knoten[b]] for a, b in muster)
                    if kosten < beste_kosten:
                        beste_kosten, beste_muster = kosten, muster
                if beste_muster is not None:
                    for i, (a, b) in zip(auswahl, beste_muster):
                        paare[i] = [knoten[a], knoten[b]]
                    verbessert = True
    return paare


def abdeckung(routen):
    """Wie oft wird jede Straße insgesamt befahren?"""
    zaehler = {}
    for r in routen:
        for a, b in zip(r, r[1:]):
            k = (min(a, b), max(a, b))
            zaehler[k] = zaehler.get(k, 0) + 1
    return zaehler


def kuerze_enden(routen, laenge, d0, vor0):
    """Ersetzt Anfang und Ende jeder Tagesroute durch einen kürzesten Weg von/zur Zentrale, wenn das kürzer ist und
    keine Straße unbefahren bleibt."""
    routen = [r[:] for r in routen]
    zaehler = abdeckung(routen)
    for tag in range(len(routen)):
        for seite in (0, 1):
            r = routen[tag] if seite == 0 else routen[tag][::-1]
            if len(r) < 3:
                continue
            vorn = [0]                              # vorn[i]: Länge des Anfangs bis Position i
            for a, b in zip(r, r[1:]):
                vorn.append(vorn[-1] + laenge[(a, b)])
            ersparnis = sorted((d0[r[i]] - vorn[i], i) for i in range(1, len(r) - 1) if d0[r[i]] < vorn[i])
            for _, i in ersparnis[:5]:
                weg_neu = weg(vor0, 0, r[i])
                ab, zu = {}, {}
                for a, b in zip(r[:i + 1], r[1:i + 1]):
                    k = (min(a, b), max(a, b))
                    ab[k] = ab.get(k, 0) + 1
                for a, b in zip(weg_neu, weg_neu[1:]):
                    k = (min(a, b), max(a, b))
                    zu[k] = zu.get(k, 0) + 1
                if all(zaehler[k] - ab[k] + zu.get(k, 0) >= 1 for k in ab):
                    for k, anzahl in ab.items():
                        zaehler[k] -= anzahl
                    for k, anzahl in zu.items():
                        zaehler[k] = zaehler.get(k, 0) + anzahl
                    r = weg_neu + r[i + 1:]
                    break
            routen[tag] = r if seite == 0 else r[::-1]
    return routen


_n, _kanten = lies(5)
_nachbarn, _laenge = straßenplan(_n, _kanten)
_ung = ungerade_knoten(_nachbarn)
_dist, _ = abstaende_der_ungeraden(_nachbarn)
_gierig = paarung_gierig(_ung, _dist)
print('muellabfuhr5.txt: %d ungerade Kreuzungen, Zusatzlänge gierig %d, nach dem Tauschen %d' % (
    len(_ung), paarung_kosten(_gierig, _dist), paarung_kosten(verbessere_paarung(_gierig, _dist), _dist)))

3 Möglichkeiten für 4 Knoten, 15 für 6 Knoten
muellabfuhr5.txt: 24 ungerade Kreuzungen, Zusatzlänge gierig 17, nach dem Tauschen 15


## Schritt 9: Alle Beispiele

Die verbesserte Version nimmt 20 zufällige Eulerkreise mit je 10 Anfangspunkten, die gute Paarung und das Kürzen der Enden. Als
**untere Schranke** nehmen wir das Größere von (Länge des Rundwegs mit bester Paarung) / 5, aufgerundet, und der Tagesstrecke, die
mindestens nötig ist, um die am weitesten entfernte Straße zu erreichen und zurückzukehren (Weg hin + Straße + Weg zurück).

In [ ]:
def untere_schranke(nr, rundlaenge):
    n, kanten = lies(nr)
    nachbarn, laenge = straßenplan(n, kanten)
    d0, _ = dijkstra(nachbarn, 0)
    entfernteste = max(min(d0[u] + l + d0[v], d0[v] + l + d0[u]) for u, v, l in kanten)
    return max(-(-rundlaenge // 5), entfernteste)


besser, schranke = {}, {}
plaene = {}
for nr in range(9):
    routen, rundlaenge = loese(nr, versuche=20, drehungen=10, kuerzen=True, paarung='gut')
    n, kanten = lies(nr)
    plaene[nr] = routen
    besser[nr] = pruefe(routen, kanten, straßenplan(n, kanten)[1])
    schranke[nr] = untere_schranke(nr, rundlaenge)
    print('muellabfuhr%d.txt: längste Tagesstrecke %s, untere Schranke %d' % (nr, besser[nr], schranke[nr]))

muellabfuhr0.txt: längste Tagesstrecke 4, untere Schranke 4
muellabfuhr1.txt: längste Tagesstrecke 18, untere Schranke 18
muellabfuhr2.txt: längste Tagesstrecke 10, untere Schranke 8
muellabfuhr3.txt: längste Tagesstrecke 22, untere Schranke 21
muellabfuhr4.txt: längste Tagesstrecke 10, untere Schranke 10
muellabfuhr5.txt: längste Tagesstrecke 1469, untere Schranke 1464
muellabfuhr6.txt: längste Tagesstrecke 540243, untere Schranke 477419
muellabfuhr7.txt: längste Tagesstrecke 794793, untere Schranke 785264
muellabfuhr8.txt: längste Tagesstrecke 2715786, untere Schranke 2666041


Vergleich mit den Lösungshinweisen (längste Tagesstrecke; in den Hinweisen ist ILP die optimale Lösung, H1 und H2 das Verfahren
dieses Notebooks mit bester bzw. gieriger Paarung, G1 und G3 einfachere gierige Verfahren; Zahlen mit Punkt als Tausendertrenner):

| Beispiel | einfach (Schritt 7) | verbessert (Schritt 9) | untere Schranke (wir) | untere Schranke (Hinweise) | ILP (Hinweise) | H1 | H2 | G3 | G1 |
|----------|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| `muellabfuhr0` | 4 | 4 | 4 | 4 | optimal 4 | 4 | 4 | 4 | 4 |
| `muellabfuhr1` | 21 | 18 | 18 | 18 | optimal 18 | 18 | 18 | 18 | 25 |
| `muellabfuhr2` | 11 | 10 | 8 | 9 | optimal 9 | 10 | 10 | 10 | 14 |
| `muellabfuhr3` | 23 | 22 | 21 | 21 | optimal 21 | 22 | 22 | 22 | 24 |
| `muellabfuhr4` | 10 | 10 | 10 | 10 | optimal 10 | 10 | 10 | 10 | 10 |
| `muellabfuhr5` | 1.471 | 1.469 | 1.464 | 1.464 | n/a | 1.468 | 1.468 | 1.468 | 1.479 |
| `muellabfuhr6` | 580.943 | 540.243 | 477.419 | 477.419 | n/a | 525.291 | 558.100 | 618.121 | 647.578 |
| `muellabfuhr7` | 795.073 | 794.793 | 785.264 | 447.294 | n/a | 794.793 | 799.288 | 813.009 | 1.236.244 |
| `muellabfuhr8` | 2.780.226 | 2.715.786 | 2.666.041 | 2.666.041 | n/a | 2.719.313 | 2.719.313 | 3.122.164 | 3.355.168 |

**Was fällt auf?**

- Die Verbesserungen helfen deutlich: bei `muellabfuhr1`, `2`, `3` und `6` ist die verbesserte Version spürbar besser als die
  einfache (z. B. 21 auf 18 bei `muellabfuhr1`, 580.943 auf 540.243 bei `muellabfuhr6`).
- Bei `muellabfuhr0`, `1` und `4` erreichen wir das Optimum der Hinweise (4, 18, 10); bei `muellabfuhr2` und `3` liegen wir 1 über dem
  Optimum (10 statt 9, 22 statt 21). Bei `muellabfuhr3` haben alle Kreuzungen geraden Grad: Der Eulerkreis ist 105 lang, ein Fünftel davon wären genau 21, und
  das Optimum ist tatsächlich 21. Wir zerlegen aber nur **einen** Rundweg in Stücke, die Stücke müssen genau aneinander passen. Der
  ILP-Fahrplan darf die Straßen beliebig auf fünf eigene Rundwege durch die Zentrale verteilen. Das ist die Grenze unseres
  Verfahrens, die auch die Hinweise nennen: Die beste Lösung muss sich nicht aus einem Gesamtrundweg zerlegen lassen.
- Gegen die Verfahren H1 und H2 der Hinweise schneiden wir ähnlich ab: gleich oder besser außer bei `muellabfuhr5` (1469 statt 1468) und
  `muellabfuhr6` (540.243 statt 525.291 bei H1, also etwa 3 % schlechter). Bei `muellabfuhr8` sind wir mit 2.715.786 etwas
  besser als H1 (2.719.313), bei `muellabfuhr7` ergibt sich dieselbe Zahl wie bei H1. Woran der Unterschied bei `muellabfuhr6` liegt,
  haben wir nicht untersucht (die Hinweise nennen als Verbesserung noch, den ersten Tagesabschnitt zu verschieben).
- Die einfachen gierigen Verfahren G1 bis G3 der Hinweise sind bei `muellabfuhr6` bis `8` deutlich schlechter als unser Verfahren (G3 z. B.
  618.121 statt 540.243 bei `muellabfuhr6`); bei `muellabfuhr5` sind G2 und G3 mit 1468 um 1 besser als wir.
- Die **untere Schranke** ist bei `muellabfuhr0`, `1`, `3`, `4`, `5`, `6` und `8` mit den Hinweisen gleich, bei `muellabfuhr2` ist die der
  Hinweise mit 9 schärfer als unsere 8 (sie stammt dort vermutlich aus der ILP-Rechnung). Bei `muellabfuhr7` ist unsere Schranke viel
  größer als die der Hinweise (785.264 statt 447.294): Dort bestimmt die am weitesten entfernte Straße die Schranke, nicht die
  Gesamtlänge. Damit liegt unser Ergebnis höchstens 0,3 % (`muellabfuhr5`), 1,2 % (`muellabfuhr7`) und 1,9 % (`muellabfuhr8`) über dem
  Optimum. Bei `muellabfuhr6` sind es höchstens 13 %; die Schranke ist dort vermutlich nicht erreichbar, weil es nur 204 Straßen mit sehr
  unterschiedlichen Längen gibt und sich die Stücke nur an Kreuzungen trennen lassen.

Der vollständige Fahrplan für `muellabfuhr1`, das erste Beispiel mit Zusatzwegen:

In [ ]:
n1, kanten1 = lies(1)
zeige(plaene[1], straßenplan(n1, kanten1)[1])

Tag 1: 0 -> 4 -> 7 -> 6 -> 0, Gesamtlaenge: 16
Tag 2: 0 -> 6 -> 7 -> 5 -> 4 -> 3 -> 6 -> 0, Gesamtlaenge: 18
Tag 3: 0 -> 5 -> 3 -> 6 -> 0, Gesamtlaenge: 11
Tag 4: 0 -> 6 -> 3 -> 2 -> 3 -> 6 -> 0, Gesamtlaenge: 18
Tag 5: 0 -> 6 -> 3 -> 1 -> 6 -> 0, Gesamtlaenge: 13
Maximale Laenge einer Tagestour: 18


Für `muellabfuhr2` und `muellabfuhr3` (lange Zeilen gekürzt):

In [ ]:
for nr in (2, 3):
    n, kanten = lies(nr)
    print('muellabfuhr%d.txt' % nr)
    for tag, r in enumerate(plaene[nr], 1):
        text = ' -> '.join(str(k) for k in r)
        if len(text) > 70:
            text = text[:67] + '...'
        print('Tag %d: %s, Gesamtlaenge: %d' % (tag, text, weglaenge(r, straßenplan(n, kanten)[1])))
    print()

muellabfuhr2.txt
Tag 1: 0 -> 9 -> 10 -> 14 -> 8 -> 7 -> 14 -> 5 -> 0 -> 6 -> 0, Gesamtlaenge: 10
Tag 2: 0 -> 6 -> 1 -> 12 -> 8 -> 11 -> 5 -> 9 -> 12 -> 9 -> 0, Gesamtlaenge: 10
Tag 3: 0 -> 9 -> 6 -> 4 -> 13 -> 14 -> 13 -> 1 -> 7 -> 9 -> 0, Gesamtlaenge: 10
Tag 4: 0 -> 9 -> 13 -> 3 -> 4 -> 10 -> 2 -> 8 -> 7 -> 9 -> 0, Gesamtlaenge: 10
Tag 5: 0 -> 9 -> 7 -> 11 -> 3 -> 11 -> 2 -> 14 -> 6 -> 0, Gesamtlaenge: 9

muellabfuhr3.txt
Tag 1: 0 -> 4 -> 2 -> 1 -> 6 -> 11 -> 13 -> 8 -> 0 -> 5 -> 2 -> 7 -> 14 ->..., Gesamtlaenge: 22
Tag 2: 0 -> 7 -> 13 -> 0 -> 12 -> 10 -> 14 -> 5 -> 4 -> 8 -> 5 -> 13 -> 12..., Gesamtlaenge: 22
Tag 3: 0 -> 11 -> 14 -> 13 -> 10 -> 7 -> 9 -> 8 -> 10 -> 9 -> 4 -> 11 -> 3..., Gesamtlaenge: 21
Tag 4: 0 -> 7 -> 5 -> 11 -> 8 -> 6 -> 9 -> 13 -> 1 -> 12 -> 7 -> 8 -> 2 ->..., Gesamtlaenge: 22
Tag 5: 0 -> 1 -> 4 -> 12 -> 6 -> 10 -> 5 -> 6 -> 4 -> 14 -> 8 -> 12 -> 9 -..., Gesamtlaenge: 22



Bei den großen Beispielen drucken wir die Wege nicht aus. Die Funktion `pruefe` hat für **alle** neun Beispiele bestätigt, dass jede
Straße befahren wird und jede Route in der Zentrale beginnt und endet.

## Zum Schluss: Geht das mit MiniZinc?

Ja, als Modell für ein ganzzahliges Programm, so wie die Hinweise es mit ILP vorschlagen. Für jede Straße `e` und jeden Tag `t` sagt
`x[e, t]` (0, 1 oder 2), wie oft die Straße an diesem Tag befahren wird. Dann gilt:

- **Jede Straße kommt vor:** die Summe über die Tage ist mindestens 1.
- **Ein Rundweg existiert:** An jedem Tag hat jede Kreuzung eine gerade Zahl benutzter Straßenenden (`2 * halb[v, t]`), und alle benutzten
  Kreuzungen hängen mit der Zentrale zusammen. Gerade Grade plus Zusammenhang sind genau die Bedingung für einen Eulerkreis durch die
  Zentrale.
- **Zusammenhang:** Jede Kreuzung bekommt eine `stufe`. Die Zentrale hat Stufe 0, und jede benutzte Kreuzung `v` muss eine benutzte
  Straße zu einer Nachbarkreuzung mit **kleinerer** Stufe haben. Dann führt von `v` ein Weg über lauter benutzte Straßen mit sinkender
  Stufe, und der endet in der Zentrale.
- **Ziel:** Die Tageslängen sind `sum(len[e] * x[e, t])`; `laengste` ist größer oder gleich allen, und wir minimieren `laengste`.
- **Symmetrie:** Die Tage sind austauschbar, wir verlangen deshalb, dass die Tageslängen absteigend sind. Das spart dem Solver viel
  Arbeit.

Neu gegenüber den Arbeitsblättern: `exists(...)`, `if ... then ... else ... endif` als Index (die andere Kreuzung einer Straße), der
Implikationspfeil `->` und Variablen mit zwei Indizes (`x[e, t]`).

```minizinc
int: n;                                 % Kreuzungen 1..n (die Zentrale ist 1)
int: m;                                 % Straßen
array[1..m] of 1..n: von;
array[1..m] of 1..n: nach;
array[1..m] of int: len;
int: T = 5;                             % Tage
array[1..m, 1..T] of var 0..2: x;       % wie oft Straße e an Tag t befahren wird
array[1..n, 1..T] of var 0..n: stufe;   % Stufe im "Erreichbarkeitsbaum" der Zentrale
array[1..n, 1..T] of var 0..n: halb;    % halber Grad der Kreuzung an diesem Tag
var int: laengste;

constraint forall(e in 1..m)(sum(t in 1..T)(x[e,t]) >= 1);
constraint forall(v in 1..n, t in 1..T)(
  sum(e in 1..m where von[e] = v \/ nach[e] = v)(x[e,t]) = 2 * halb[v,t]);
constraint forall(t in 1..T)(stufe[1,t] = 0);
constraint forall(v in 2..n, t in 1..T)(
  halb[v,t] > 0 -> exists(e in 1..m)(
    (von[e] = v \/ nach[e] = v) /\ x[e,t] > 0 /\
    stufe[if von[e] = v then nach[e] else von[e] endif, t] < stufe[v,t]));
constraint forall(t in 1..T)(sum(e in 1..m)(len[e] * x[e,t]) <= laengste);
constraint forall(t in 1..T-1)(
  sum(e in 1..m)(len[e] * x[e,t]) >= sum(e in 1..m)(len[e] * x[e,t+1]));
solve minimize laengste;
output ["laengste = ", show(laengste), "\n"];
```

Die Datendatei erzeugt die Python-Zelle `als_dzn` (MiniZinc zählt ab 1, deshalb `+ 1`).

In [ ]:
def als_dzn(nr):
    n, kanten = lies(nr)
    return 'n = %d;\nm = %d;\nvon = %s;\nnach = %s;\nlen = %s;' % (
        n, len(kanten), [u + 1 for u, v, l in kanten], [v + 1 for u, v, l in kanten], [l for u, v, l in kanten])


print(als_dzn(0))

n = 10;
m = 13;
von = [1, 1, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9];
nach = [3, 5, 7, 9, 3, 4, 5, 6, 7, 8, 9, 2, 10];
len = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1];


Ergebnis beim Erzeugen dieses Notebooks (Zeitlimit 20 Sekunden für Gecode, 60 bzw. 30 Sekunden für Chuffed; die Solver wählt man in
MiniZinc über *Solver configuration*). Eine Zahl heißt: Der Solver hat diesen Wert **bewiesen optimal** gefunden (er ist gleich
dem Ergebnis des Python-Programms). „?“ heißt: Zeitlimit, nicht fertig. Die großen Beispiele `muellabfuhr5` bis `muellabfuhr8` haben
wir gar nicht erst versucht; die Hinweise schreiben, dass ihr ILP-Ansatz sie nicht in vernünftiger Zeit löst.

| Beispiel | Gecode | Chuffed | Python (Schritt 9) | ILP in den Hinweisen |
|----------|--------|---------|--------------------|----------------------|
| `muellabfuhr0` | ? | 4 | 4 | 4 |
| `muellabfuhr1` | nicht probiert | 18 | 18 | 18 |
| `muellabfuhr2` | nicht probiert | ? | 10 | 9 |
| `muellabfuhr3` | nicht probiert | ? | 22 | 21 |
| `muellabfuhr4` | ? | 10 | 10 | 10 |

Das Modell ist richtig, aber nur für ganz kleine Straßenpläne brauchbar: Mit Chuffed (der aus Konflikten lernt) sind die Pläne mit
10 bis 13 Straßen in wenigen Sekunden bewiesen, bei den Plänen mit 34 und 105 Straßen kommt innerhalb des Zeitlimits kein bewiesenes
Optimum. Dass Python bei `muellabfuhr2` und `3` 1 über dem Optimum der Hinweise liegt (10 und 22), konnten wir mit MiniZinc deshalb
nicht nachprüfen. Der Grund: Der Solver muss die Verteilung der Straßen auf fünf Tage durchprobieren (die Aufgabe ist NP-schwer),
und die Schranke aus der Relaxation ist schwach. Dafür braucht man spezialisierte ILP-Solver mit einem ausgefeilteren Modell, wie in der
Arbeit, die die Hinweise zitieren.

## Für Neugierige

- **Optimale Paarung:** Die Paarung der ungeraden Kreuzungen mit kleinster Gesamtlänge ist ein *Minimum-Cost Perfect Matching* in
  einem allgemeinen Graphen; dafür gibt es den **Blossom-Algorithmus** (Edmonds), in Python z. B. `networkx.min_weight_matching`
  (nicht in JupyterLite). Beim Erzeugen dieses Notebooks haben wir damit geprüft, dass unser Tauschen von zwei und drei Paaren bei allen
  Beispielen die optimale Zusatzlänge findet. Die untere Schranke oben ist deshalb gültig.
- **Aufteilen des Rundwegs genauer:** Unsere Zerlegung nimmt immer das längste mögliche Stück. Die Hinweise schlagen vor, auch den
  Anfangspunkt zu verschieben; wir probieren dafür zufällige Drehungen des Rundwegs.
- **Enden:** Tagesenden kürzen wir nur durch kürzeste Wege zur Zentrale. Mit einer genaueren Suche ließe sich noch etwas herausholen.
- **Laufzeit:** Dijkstra von jeder ungeraden Kreuzung, Hierholzer in `O(Kanten)`, die binäre Suche mit etwa `log(Länge)` Zerlegungen
  zu je `O(Kanten)`. Das Tauschen von drei Paaren ist bei vielen ungeraden Kreuzungen (`muellabfuhr7` hat 240) der teuerste Teil.